# Carga de librerías

In [ ]:
# --- Setup / Imports ---
import os, json, time, random, shutil, datetime
from pathlib import Path

import numpy as np
import cv2

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from torch.optim import AdamW, Adam
from torch.optim.lr_scheduler import CosineAnnealingLR, SequentialLR, LinearLR
from torch.amp import autocast, GradScaler

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


# Configuración y carga de datos

In [ ]:
# --- Config ---
class Cfg:
    data_root = '/home/joan_ds/Sandbox/UOC/TFM/data/dataset_500_GT'  # TODO: set your path
    work_dir  = '/home/joan_ds/Sandbox/UOC/TFM/train_DeepLabV3'
    mode      = 'labelme'  # 'labelme' or 'png'
    cityscapes_ckpt_for_ft = ''
    img_size  = (768, 768)
    batch_size  = 2
    num_workers = 2
    cache_png_masks = True
    CLASS_TO_INDEX = {
        'sidewalk_tiles': 0,
        'sidewalk_asphalt': 1,
        'roadway': 2,
        'curb_edge': 3,
        'drainage_inlet': 4,
        'gutter': 5,
        'access_cover': 6,
        'tree_pit': 7,
        'vegetation': 8,
        'street_furniture': 9
    }
    ignore_index = 255
    epochs_grid  = [90, 135]
    lr_base_grid = [1e-4]
    opt_grid     = ['AdamW']

cfg = Cfg()
cfg.num_classes = len(cfg.CLASS_TO_INDEX)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
IDX_TO_CLASS = {v: k for k, v in cfg.CLASS_TO_INDEX.items()}
print('Classes:', cfg.CLASS_TO_INDEX)
print('Work dir:', cfg.work_dir)


In [ ]:
# --- Label normalization, Rasterization (LabelMe) ---
PRIORITY = [
    'sidewalk_tiles','sidewalk_asphalt','roadway','curb_edge','drainage_inlet',
    'gutter','access_cover','tree_pit','vegetation','street_furniture'
]

ALIASES = {
    'street_forniture': 'street_furniture',
    'kerb': 'curb_edge',
    'curb': 'curb_edge',
    'gully': 'drainage_inlet',
    'drain': 'drainage_inlet',
    'inlet': 'drainage_inlet',
    'grate': 'drainage_inlet',
    'manhole': 'access_cover',
    'manhole_cover': 'access_cover',
    'panot': 'sidewalk_tiles',
    'asphalt': 'sidewalk_asphalt',
    'asphalt_sidewalk': 'sidewalk_asphalt',
    'bush': 'vegetation',
    'tree': 'vegetation',
    'road': 'roadway',
    'street': 'roadway',
}

def norm_label(raw):
    s = str(raw).strip().lower()
    if s in ('255','ignore','ignored','void','background','bg'):
        return None
    if s in ('0','1','2','3','4','5','6','7','8','9'):
        idx = int(s)
        return IDX_TO_CLASS.get(idx, None)
    if s in ALIASES:
        return ALIASES[s]
    return s

def draw_polygons(mask: np.ndarray, polygons: list, value: int):
    if not polygons: return
    cnts = []
    for poly in polygons:
        if len(poly) < 3: 
            continue
        arr = np.asarray(poly, dtype=np.float32)
        arr = np.round(arr).astype(np.int32)
        cnts.append(arr.reshape(-1, 1, 2))
    if cnts:
        cv2.fillPoly(mask, cnts, color=int(value))

def labelme_to_mask(json_path: Path, image_hw: tuple, class_to_index: dict, ignore_val: int):
    H, W = image_hw
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    h = data.get('imageHeight', H)
    w = data.get('imageWidth', W)
    mask = np.full((h, w), ignore_val, dtype=np.uint8)

    by_label = {lab: [] for lab in PRIORITY}
    for shp in data.get('shapes', []):
        lab = norm_label(shp.get('label'))
        if lab in class_to_index:
            pts = shp.get('points', [])
            by_label[lab].append(pts)
    for lab in PRIORITY:
        if by_label.get(lab):
            draw_polygons(mask, by_label[lab], class_to_index[lab])
    return mask


In [ ]:
# --- Dataset class (LabelMe or PNG) ---
class SegDataset(Dataset):
    def __init__(self, root, split='train', mode='labelme', img_size=(768,768), augment=False, cache=False, class_to_index=None, ignore_val=255):
        self.root = Path(root)
        self.split = split
        self.mode = mode
        self.img_dir = self.root / 'images' / split
        self.seg_dir = self.root / ('segmaps' if mode=='png' else 'labelme') / split
        self.imgs = sorted([p for p in self.img_dir.glob('*') if p.suffix.lower() in ['.jpg','.jpeg','.png','.bmp']])
        self.size = img_size
        self.augment = augment
        self.class_to_index = class_to_index or {}
        self.ignore_val = ignore_val

        self.cache = cache and (mode=='labelme')
        self.cache_dir = self.root / '_cache_png' / split if self.cache else None
        if self.cache_dir: self.cache_dir.mkdir(parents=True, exist_ok=True)

        self.norm = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        name = img_path.stem

        img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        assert img is not None, f"Image not found: {img_path}"
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.mode == 'png':
            seg_path = self.seg_dir / f'{name}.png'
            seg = cv2.imread(str(seg_path), cv2.IMREAD_UNCHANGED)
            assert seg is not None, f"Mask not found: {seg_path}"
            if seg.ndim == 3:
                seg = seg[:,:,0]
        else:
            cache_path = self.cache_dir / f'{name}.png' if self.cache_dir else None
            if cache_path and cache_path.exists():
                seg = cv2.imread(str(cache_path), cv2.IMREAD_UNCHANGED)
            else:
                json_path = self.seg_dir / f'{name}.json'
                assert json_path.exists(), f"LabelMe JSON not found: {json_path}"
                seg = labelme_to_mask(json_path, image_hw=img.shape[:2], class_to_index=self.class_to_index, ignore_val=self.ignore_val)
                if cache_path is not None:
                    cv2.imwrite(str(cache_path), seg)

        H, W = self.size
        img_r = cv2.resize(img, (W,H), interpolation=cv2.INTER_LINEAR)
        seg_r = cv2.resize(seg, (W,H), interpolation=cv2.INTER_NEAREST)

        if self.augment and random.random() < 0.5:
            img_r = np.ascontiguousarray(np.fliplr(img_r))
            seg_r = np.ascontiguousarray(np.fliplr(seg_r))

        img_t = torch.from_numpy(img_r).permute(2,0,1).float()/255.0
        img_t = self.norm(img_t)
        seg_t = torch.from_numpy(seg_r.astype(np.int64))

        return {'image': img_t, 'mask': seg_t, 'name': name}


In [ ]:
# --- Optional: Create train/val/test split from flat folders ---
DO_SPLIT = False
if DO_SPLIT:
    root = Path(cfg.data_root)
    images_flat = sorted([p for p in (root/'images').glob('*') if p.suffix.lower() in ['.jpg','.jpeg','.png','.bmp']])
    jsons_flat  = sorted([p for p in (root/'labelme').glob('*.json')])
    names_img = {p.stem for p in images_flat}
    names_json= {p.stem for p in jsons_flat}
    names = sorted(list(names_img & names_json))
    print('Total pairs:', len(names))
    random.seed(1337)
    random.shuffle(names)
    n = len(names)
    n_train = int(0.70*n)
    n_val   = int(0.15*n)
    splits = {'train': names[:n_train], 'val': names[n_train:n_train+n_val], 'test': names[n_train+n_val:]}
    for sp, lst in splits.items():
        (root/'images'/sp).mkdir(parents=True, exist_ok=True)
        (root/'labelme'/sp).mkdir(parents=True, exist_ok=True)
        for nm in lst:
            for ext in ['.jpg','.jpeg','.png','.bmp']:
                src = root/'images'/f'{nm}{ext}'
                if src.exists():
                    dst = root/'images'/sp/f'{nm}{ext}'
                    shutil.copy2(src, dst)
                    break
            srcj = root/'labelme'/f'{nm}.json'
            if srcj.exists():
                dstj = root/'labelme'/sp/f'{nm}.json'
                shutil.copy2(srcj, dstj)
    print('Split created.')


In [ ]:
# --- Optional: Bulk rasterization from LabelMe to PNG ---
DO_RASTERIZE = False
if DO_RASTERIZE:
    root = Path(cfg.data_root)
    for sp in ['train','val','test']:
        img_dir = root/'images'/sp
        json_dir= root/'labelme'/sp
        out_dir = root/'segmaps'/sp
        out_dir.mkdir(parents=True, exist_ok=True)
        count = 0
        for img_path in sorted(img_dir.glob('*')):
            if img_path.suffix.lower() not in ['.jpg','.jpeg','.png','.bmp']:
                continue
            name = img_path.stem
            json_path = json_dir/f'{name}.json'
            if not json_path.exists():
                continue
            img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
            mask = labelme_to_mask(json_path, image_hw=img.shape[:2], class_to_index=cfg.CLASS_TO_INDEX, ignore_val=cfg.ignore_index)
            cv2.imwrite(str(out_dir/f'{name}.png'), mask)
            count += 1
        print(f'[Rasterize {sp}] wrote {count} PNG masks to {out_dir}')


In [ ]:
from pathlib import Path

root = Path(cfg.data_root)
print("data_root =", root.resolve())
print("mode     =", cfg.mode)

def ls(p):
    return sorted([q.name for q in p.glob("*") if q.is_file()])

for sp in ["train","val","test"]:
    img_dir = root/"images"/sp
    if cfg.mode == "png":
        seg_dir = root/"segmaps"/sp
    else:
        seg_dir = root/"labelme"/sp

    print(f"\n[{sp}]")
    print(" images dir:", img_dir,  "exists:", img_dir.exists())
    print(" ann dir   :", seg_dir,   "exists:", seg_dir.exists())

    imgs = [p for p in img_dir.glob("*") if p.suffix.lower() in [".jpg",".jpeg",".png",".bmp"]]
    if cfg.mode == "png":
        anns = [p for p in seg_dir.glob("*.png")]
    else:
        anns = [p for p in seg_dir.glob("*.json")]

    print(" #images =", len(imgs))
    print(" #annots =", len(anns))

    # Comprovem quants noms coincideixen (stem)
    img_stems = {p.stem for p in imgs}
    ann_stems = {p.stem for p in anns}
    inter = sorted(img_stems & ann_stems)
    print(" #matching pairs =", len(inter))
    # Mostra algun exemple
    print(" sample matches:", inter[:5])
    print(" sample missing images for ann:", sorted(ann_stems - img_stems)[:5])
    print(" sample missing ann for images:", sorted(img_stems - ann_stems)[:5])

In [ ]:
# --- Build DataLoaders ---
def make_loaders(cfg):
    ds_tr = SegDataset(cfg.data_root, 'train', cfg.mode, cfg.img_size, augment=True,  cache=cfg.cache_png_masks, class_to_index=cfg.CLASS_TO_INDEX, ignore_val=cfg.ignore_index)
    ds_va = SegDataset(cfg.data_root, 'val',   cfg.mode, cfg.img_size, augment=False, cache=cfg.cache_png_masks, class_to_index=cfg.CLASS_TO_INDEX, ignore_val=cfg.ignore_index)
    dl_tr = DataLoader(ds_tr, batch_size=cfg.batch_size, shuffle=True,  num_workers=cfg.num_workers, pin_memory=True)
    dl_va = DataLoader(ds_va, batch_size=1,               shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    return ds_tr, ds_va, dl_tr, dl_va

ds_tr, ds_va, dl_tr, dl_va = make_loaders(cfg)
print(f'Train: {len(ds_tr)}   Val: {len(ds_va)}   Mode: {cfg.mode}')


# Training

In [ ]:
# --- Training runner (grid) amb EarlyStopping per val_loss (patience=20) ---
def set_seed(seed=1337):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

def build_model(num_classes):
    return models.segmentation.deeplabv3_resnet50(weights=None, aux_loss=True, num_classes=num_classes)

def split_param_groups(model, lr_base, head_mult=10.0, weight_decay=1e-4):
    bb, head = [], []
    for n,p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head if ('classifier' in n or 'aux_classifier' in n) else bb).append(p)
    return [
        {'params': bb,   'lr': lr_base,           'weight_decay': weight_decay},
        {'params': head, 'lr': lr_base*head_mult, 'weight_decay': weight_decay},
    ]

def build_optimizer(name, model, lr_base, wd, head_mult=10.0):
    param_groups = split_param_groups(model, lr_base, head_mult=head_mult, weight_decay=wd)
    name = name.lower()
    if name == 'adamw':
        return AdamW(param_groups, betas=(0.9, 0.999), eps=1e-8)
    elif name == 'adam':
        return Adam(param_groups, betas=(0.9, 0.999), eps=1e-8)
    else:
        raise ValueError(f'Unsupported optimizer: {name}')

def build_scheduler(optimizer, total_epochs, warmup_epochs=5):
    warmup_epochs = min(warmup_epochs, max(1, total_epochs//10))
    return SequentialLR(
        optimizer,
        schedulers=[
            LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs),
            CosineAnnealingLR(optimizer, T_max=total_epochs - warmup_epochs)
        ],
        milestones=[warmup_epochs]
    )

@torch.no_grad()
def validate_loss_and_miou(model, dl, device, cfg, criterion):
    model.eval()
    nC = cfg.num_classes
    conf = np.zeros((nC, nC), dtype=np.int64)
    loss_acc = 0.0
    count = 0
    for batch in dl:
        x = batch['image'].to(device)
        y = batch['mask'].to(device)
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            out = model(x)['out']
            loss = criterion(out, y)
        loss_acc += float(loss.item())
        count += 1
        pred = out.argmax(1)
        gt = y
        mask = (gt != cfg.ignore_index)
        if mask.sum().item() == 0:
            continue
        gt_v = gt[mask].view(-1).cpu().numpy()
        pr_v = pred[mask].view(-1).cpu().numpy()
        cm = np.bincount(gt_v * nC + pr_v, minlength=nC*nC)
        conf += cm.reshape(nC, nC)

    val_loss = loss_acc / max(1, count)
    inter = np.diag(conf).astype(np.float64)
    gt_sum = conf.sum(axis=1)
    pr_sum = conf.sum(axis=0)
    union = gt_sum + pr_sum - inter
    iou = inter / np.maximum(union, 1e-7)
    miou = float(np.nanmean(iou))
    return val_loss, miou, iou.tolist()

def class_index_to_names(iou_list, class_to_index):
    inv = [None]*len(class_to_index)
    for k,v in class_to_index.items():
        inv[v] = k
    return {inv[i]: float(iou_list[i]) for i in range(len(inv))}

def train_one_exp(optimizer_name, epochs, lr_base, cfg, dl_tr, dl_va, device, seed=1337, early_patience=20):
    set_seed(seed)
    wd = 1e-5
    head_mult = 10.0

    tag = f"{optimizer_name}_e{epochs}_lr{lr_base:g}_wd{wd:g}"
    exp_dir = Path(cfg.work_dir) / f"exp_{tag}"
    exp_dir.mkdir(parents=True, exist_ok=True)
    best_path   = exp_dir / 'best.pth'
    log_path    = exp_dir / 'train_log.jsonl'
    summary_path= exp_dir / 'summary.json'

    model = models.segmentation.deeplabv3_resnet50(weights=None, aux_loss=True, num_classes=cfg.num_classes).to(device)

    if getattr(cfg, 'cityscapes_ckpt_for_ft', '') and Path(cfg.cityscapes_ckpt_for_ft).exists():
        state = torch.load(cfg.cityscapes_ckpt_for_ft, map_location='cpu')
        msd = model.state_dict()
        filt = {k:v for k,v in state.items() if k in msd and msd[k].shape == v.shape}
        model.load_state_dict(filt, strict=False)

    optim = build_optimizer(optimizer_name, model, lr_base, wd, head_mult=head_mult)
    scheduler = build_scheduler(optim, total_epochs=epochs, warmup_epochs=5)
    scaler = GradScaler('cuda', True)
    criterion = nn.CrossEntropyLoss(ignore_index=cfg.ignore_index)

    best_miou = -1.0
    best_per_class_iou = None
    best_epoch = None

    # EarlyStopping state
    best_val_loss = float('inf')
    no_improve = 0

    epoch_train_losses = []
    t0 = time.time()

    with open(log_path, 'w') as f_log:
        for ep in range(1, epochs+1):
            model.train()
            running = 0.0
            for batch in dl_tr:
                x = batch['image'].to(device); y = batch['mask'].to(device)
                optim.zero_grad(set_to_none=True)
                with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                    out = model(x)['out']
                    loss = criterion(out, y)
                scaler.scale(loss).backward()
                scaler.step(optim); scaler.update()
                running += loss.item()
            scheduler.step()
            tr_loss = running / max(1, len(dl_tr))
            epoch_train_losses.append(tr_loss)

            # Validació: loss + mIoU
            val_loss, miou_va, ious_va = validate_loss_and_miou(model, dl_va, device, cfg, criterion)

            # Best by mIoU (checkpoint)
            if miou_va > best_miou:
                best_miou = miou_va
                best_per_class_iou = ious_va
                best_epoch = ep
                torch.save(model.state_dict(), best_path)

            # EarlyStopping per val_loss
            if val_loss < best_val_loss - 1e-8:
                best_val_loss = val_loss
                no_improve = 0
            else:
                no_improve += 1

            rec = {
                'epoch': ep, 'optimizer': optimizer_name,
                'train_loss': tr_loss, 'val_loss': val_loss,
                'val_mIoU': miou_va, 'best_mIoU': best_miou, 'best_epoch': best_epoch,
                'lr_bb': optim.param_groups[0]['lr'], 'lr_head': optim.param_groups[1]['lr'],
                'weight_decay': wd, 'early_no_improve': no_improve, 'early_best_val_loss': best_val_loss
            }
            f_log.write(json.dumps(rec) + '\n'); f_log.flush()
            print(f"[{tag}] Ep {ep:03d}/{epochs}  tr_loss={tr_loss:.3f}  val_loss={val_loss:.3f}  mIoU={miou_va:.4f}  best_mIoU={best_miou:.4f} (ep={best_epoch})  ES({no_improve}/{early_patience})")

            if no_improve >= early_patience:
                print(f">>> EarlyStopping activat a l'època {ep} (val_loss no millora en {early_patience} èpoques).")
                break

    duration_sec = time.time() - t0
    per_class = class_index_to_names(best_per_class_iou, cfg.CLASS_TO_INDEX) if best_per_class_iou is not None else {}
    summary = {
        'tag': tag,
        'paths': {
            'exp_dir': str(exp_dir),
            'best_ckpt': str(best_path),
            'train_log': str(log_path)
        },
        'hyperparams': {
            'optimizer': optimizer_name,
            'epochs': epochs,
            'lr_backbone': lr_base,
            'lr_head_multiplier': 10.0,
            'weight_decay': wd,
            'seed': 1337,
            'early_stopping': {'monitor': 'val_loss', 'patience': early_patience}
        },
        'duration_seconds': round(duration_sec, 2),
        'metrics': {
            'train_loss_mean': float(np.mean(epoch_train_losses)) if epoch_train_losses else None,
            'train_loss_last': float(epoch_train_losses[-1]) if epoch_train_losses else None,
            'best_val_mIoU': best_miou,
            'best_epoch': best_epoch,
            'best_val_loss': best_val_loss,
            'per_class_IoU': per_class
        }
    }
    with open(summary_path, 'w') as fsum:
        json.dump(summary, fsum, indent=2)
    return summary



In [ ]:

epochs_grid  = cfg.epochs_grid
lr_base_grid = cfg.lr_base_grid
opt_grid     = cfg.opt_grid

all_summaries = []
best_overall = {'best_val_mIoU': -1.0, 'best_ckpt': None, 'best_epoch': None, 'summary': None}

for opt_name in opt_grid:
    for ep in epochs_grid:
        for lr in lr_base_grid:
            summary = train_one_exp(opt_name, ep, lr, cfg, dl_tr, dl_va, device, seed=1337, early_patience=20)
            all_summaries.append(summary)
            if summary['metrics']['best_val_mIoU'] > best_overall['best_val_mIoU']:
                best_overall = {
                    'best_val_mIoU': summary['metrics']['best_val_mIoU'],
                    'best_ckpt': summary['paths']['best_ckpt'],
                    'best_epoch': summary['metrics']['best_epoch'],
                    'summary': summary
                }

work_dir = Path(cfg.work_dir)
work_dir.mkdir(parents=True, exist_ok=True)

results_path = work_dir / 'results_all.json'
with open(results_path, 'w') as f:
    json.dump({
        'run_date': datetime.date.today().isoformat(),
        'experiments': all_summaries,
        'best_overall': best_overall
    }, f, indent=2)

if best_overall['best_ckpt'] is not None:
    dated_best = work_dir / f"{datetime.date.today().isoformat()}_best.pth"
    shutil.copy2(best_overall['best_ckpt'], dated_best)
    print(f"\n>>> Best global checkpoint copied to: {dated_best}")
    print(f"    mIoU={best_overall['best_val_mIoU']:.4f}  epoch={best_overall['best_epoch']}  (from: {best_overall['best_ckpt']})")
else:
    print("\n>>> No best checkpoint found. Check logs.")

In [ ]:
torch.cuda.empty_cache() 

# Test (visualization)

In [ ]:
CLASS_TO_INDEX = {
    "sidewalk_tiles": 0,
    "sidewalk_asphalt": 1,
    "roadway": 2,
    "curb_edge": 3,
    "drainage_inlet": 4,
    "gutter": 5,
    "access_cover": 6,
    "tree_pit": 7,
    "vegetation": 8,
    "street_furniture": 9,
}
INDEX_TO_CLASS = {v:k for v,k in enumerate(CLASS_TO_INDEX)}

PALETTE = np.array([
  [166,206,227],[31,120,180],[178,223,138],[51,160,44],[251,154,153],
  [227,26,28],[253,191,111],[255,127,0],[202,178,214],[106,61,154]
], dtype=np.uint8)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# ========= Paràmetres de mida/gràfica (grans, poc espai en blanc) ============
FIG_W_3 = 34   # amplada amb 3 panells (Input, Cityscapes, BIAA)
FIG_W_2 = 26   # amplada amb 2 panells (si no hi ha Cityscapes)
FIG_H   = 12
DPI     = 180

AX_TITLE_FONTSIZE = 30
AX_TITLE_PAD = 6

# Llegendes (grans)
LEGEND_FONTSIZE_BIAA       = 22
LEGEND_TITLE_FONTSIZE_BIAA = 24
LEGEND_FONTSIZE_CITY       = 22
LEGEND_TITLE_FONTSIZE_CITY = 24

# Marges molt compactes
RIGHT_MARGIN_HAS_CITY = 0.965
RIGHT_MARGIN_NO_CITY  = 0.970
BOTTOM_MARGIN_CITY    = 0.010

# Tight layout compacte
TIGHT_PAD   = 0.30
TIGHT_W_PAD = 0.30
TIGHT_H_PAD = 0.10

# ========= Funcions de llegenda (BIAA i Cityscapes) ==========================
import matplotlib.patches as mpatches

def class_legend_handles_biaa():
    handles = []
    for cls_name, idx in CLASS_TO_INDEX.items():
        color = (PALETTE[idx] / 255.0).tolist()
        handles.append(mpatches.Patch(color=color, label=f"{idx}: {cls_name}"))
    handles.sort(key=lambda h: int(h.get_label().split(":")[0]))
    return handles

CITY_LABELS_19 = [
    "road","sidewalk","building","wall","fence","pole","traffic light","traffic sign",
    "vegetation","terrain","sky","person","rider","car","truck","bus","train","motorcycle","bicycle"
]
CITY_PALETTE_19 = np.array([
    [178,223,138], [166,206,227], [ 70, 70, 70], [102,102,156], [190,153,153],
    [153,153,153], [250,170, 30], [220,220,  0], [107,142, 35], [152,251,152],
    [ 70,130,180], [220, 20, 60], [255,  0,  0], [  0,  0,142], [  0,  0, 70],
    [  0, 60,100], [  0,  80,100], [  0,  0,230], [119, 11, 32]
], dtype=np.uint8)

def class_legend_handles_city():
    handles = []
    for idx, name in enumerate(CITY_LABELS_19):
        color = (CITY_PALETTE_19[idx] / 255.0).tolist()
        handles.append(mpatches.Patch(color=color, label=f"{idx}: {name}"))
    return handles

# ========= Utilitats ==========================================================
from torch.amp import autocast
use_cuda = torch.cuda.is_available()

@torch.no_grad()
def colorize_idx_biaa(idx_hw):
    out = np.zeros((*idx_hw.shape, 3), np.uint8)
    valid = (idx_hw >= 0) & (idx_hw < len(PALETTE))
    out[valid] = PALETTE[idx_hw[valid]]
    return out

@torch.no_grad()
def colorize_idx_city(idx_hw):
    out = np.zeros((*idx_hw.shape, 3), np.uint8)
    valid = (idx_hw >= 0) & (idx_hw < len(CITY_PALETTE_19))
    out[valid] = CITY_PALETTE_19[idx_hw[valid]]
    return out

@torch.no_grad()
def overlay(img_rgb, mask_rgb, a=0.45):
    return np.clip((1 - a) * img_rgb + a * mask_rgb, 0, 255).astype(np.uint8)

# ========= DataLoader test (si cal) ===========================================
try:
    dl_te
except NameError:
    ds_te = SegDataset(cfg.data_root, split="test", mode=cfg.mode, img_size=cfg.img_size, augment=False)
    dl_te = DataLoader(ds_te, batch_size=1, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    print(f"Test: {len(ds_te)} imatges")

# ========= Carrega models (BIAA millor model, Cityscapes baseline) ============
# Tria automàticament el millor reentrenat: cfg.best_pth si existeix; si no, work_dir/best.pth
biaa_ckpt = "/home/joan_ds/Sandbox/UOC/TFM/train_DeepLabV3/2025-12-02_best.pth"
if not biaa_ckpt or not Path(biaa_ckpt).exists():
    biaa_ckpt = os.path.join(cfg.work_dir, "best.pth")

if Path(biaa_ckpt).exists():
    def build_for_vis_biaa():
        m = models.segmentation.deeplabv3_resnet50(weights=None, aux_loss=True, num_classes=cfg.num_classes)
        return m.to(device).eval()
    m_biaa = build_for_vis_biaa()
    m_biaa.load_state_dict(torch.load(biaa_ckpt, map_location=device))
    m_biaa.eval()
    print("BIAA (best) carregat des de:", biaa_ckpt)
else:
    raise FileNotFoundError(f"No s'ha trobat el checkpoint BIAA: {biaa_ckpt}")

# Cityscapes (19 classes) opcional
m_city = None
if Path("/home/joan_ds/Sandbox/UOC/TFM/train_DeepLabV3/pytorch_model.bin").exists():
    def build_for_vis_city():
        m = models.segmentation.deeplabv3_resnet50(weights=None, aux_loss=True, num_classes=19)
        state = torch.load("/home/joan_ds/Sandbox/UOC/TFM/train_DeepLabV3/pytorch_model.bin", map_location="cpu")
        msd = m.state_dict()
        filt = {k:v for k,v in state.items() if k in msd and msd[k].shape == v.shape}
        m.load_state_dict(filt, strict=False)
        return m.to(device).eval()
    try:
        m_city = build_for_vis_city()
        print("Cityscapes baseline carregat des de:", "/home/joan_ds/Sandbox/UOC/TFM/train_DeepLabV3/pytorch_model.bin")
    except Exception as e:
        print("No s'ha pogut carregar el model Cityscapes:", e)
        m_city = None
else:
    print("Avís: cfg.cityscapes_ckpt no especificat o inexistent; es mostrarà només BIAA.")

# ====== Recorre TOT el test: Input + Overlay(Cityscapes) + Overlay(BIAA) ======
for batch in dl_te:
    x    = batch["image"].to(device)
    name = batch["name"][0]

    # --- Predicció BIAA ---
    with autocast(device_type='cuda', enabled=use_cuda):
        out_biaa = m_biaa(x)["out"]
    pred_biaa = out_biaa.argmax(1).cpu().numpy()[0]

    # desnormalitza input
    xn  = x[0].cpu().numpy()
    img = (xn.transpose(1,2,0)*np.array([0.229,0.224,0.225]) +
           np.array([0.485,0.456,0.406]))*255.0
    img = np.clip(img, 0, 255).astype(np.uint8)

    pm_biaa   = colorize_idx_biaa(pred_biaa)
    over_biaa = overlay(img, pm_biaa, 0.45)

    # --- Predicció Cityscapes (si disponible) ---
    over_city = None
    if m_city is not None:
        with autocast(device_type='cuda', enabled=use_cuda):
            out_city = m_city(x)["out"]
        pred_city = out_city.argmax(1).cpu().numpy()[0]
        pm_city   = colorize_idx_city(pred_city)
        over_city = overlay(img, pm_city, 0.45)

    # ====== Plot: només Input + Overlay(Cityscapes) + Overlay(BIAA) ======
    cols = 3 if over_city is not None else 2
    fig_w = FIG_W_3 if cols == 3 else FIG_W_2
    plt.figure(figsize=(fig_w, FIG_H), dpi=DPI)

    # 1) Original
    plt.subplot(1, cols, 1); plt.imshow(img, interpolation="nearest")
    plt.title(f"Input\n{name}", fontsize=AX_TITLE_FONTSIZE, pad=AX_TITLE_PAD); plt.axis("off")

    # 2) Cityscapes (si hi és) — si no, BIAA al panell 2
    if cols == 3:
        plt.subplot(1, cols, 2); plt.imshow(over_city, interpolation="nearest")
        plt.title("Overlay (Cityscapes)", fontsize=AX_TITLE_FONTSIZE, pad=AX_TITLE_PAD); plt.axis("off")
        plt.subplot(1, cols, 3); plt.imshow(over_biaa, interpolation="nearest")
        plt.title("Overlay (BIAA)", fontsize=AX_TITLE_FONTSIZE, pad=AX_TITLE_PAD); plt.axis("off")
    else:
        plt.subplot(1, cols, 2); plt.imshow(over_biaa, interpolation="nearest")
        plt.title("Overlay (BIAA)", fontsize=AX_TITLE_FONTSIZE, pad=AX_TITLE_PAD); plt.axis("off")

    # compacte: poc coixí entre subplots
    plt.tight_layout(pad=TIGHT_PAD, w_pad=TIGHT_W_PAD, h_pad=TIGHT_H_PAD)

    # Llegenda BIAA a la dreta
    fig = plt.gcf()
    handles_biaa = class_legend_handles_biaa()
    fig.legend(
        handles=handles_biaa,
        loc='center left',
        bbox_to_anchor=(1.002, 0.5),
        borderaxespad=0.,
        frameon=True,
        title="Classes BIAA",
        prop={"size": LEGEND_FONTSIZE_BIAA},
        title_fontsize=LEGEND_TITLE_FONTSIZE_BIAA
    )

    # Llegenda Cityscapes a sota (si hi és)
    if cols == 3:
        handles_city = class_legend_handles_city()
        fig.legend(
            handles=handles_city,
            loc='upper center',
            bbox_to_anchor=(0.5, -0.015),
            ncol=5,
            frameon=True,
            title="Classes Cityscapes (19)",
            prop={"size": LEGEND_FONTSIZE_CITY},
            title_fontsize=LEGEND_TITLE_FONTSIZE_CITY
        )
        plt.subplots_adjust(right=RIGHT_MARGIN_HAS_CITY, bottom=BOTTOM_MARGIN_CITY)
    else:
        plt.subplots_adjust(right=RIGHT_MARGIN_NO_CITY)

    plt.show()